# វគ្គទី 6 – បណ្តាញដំណើរការ​ច្រើនជំហាន (ផែនការ → អនុវត្ត → កែលម្អ)

ប្រើ routing functions ដើម្បីជ្រើសម៉ូឌែលក្នុងនីមួយៗជំហាន បន្ទាប់មកកែលម្អចម្លើយចុងក្រោយ។


### ការពន្យល់៖ ការតំឡើងអាស្រ័យភាព
តម្លើង `foundry-local-sdk` និង `openai` ដែលចាំបាច់សម្រាប់ការហៅជជែកក្នុងរៀងរាល់ជំហាន។ អាចដំណើរការឡើងវិញបានយ៉ាងសុវត្ថិភាព។


# Scenario
ខ្សែដំណើរការម៉ូដែលច្រើនជំហានដែល៖ (1) រៀបចំភារកិច្ចជាជំហានៗច្បាស់, (2) អនុវត្តជំហាននីមួយៗដោយជ្រើសម៉ូដែលផ្អែកលើចេតនា, (3) ធ្វើឲ្យចម្លើយសម្រួលចុងក្រោយកាន់តែមានគុណភាព។ បង្ហាញពីការចងខ្សែសមត្ថភាពផ្សេងៗរបស់ SLM ដើម្បីកែលម្អប្រសិទ្ធភាព ខណៈនៅតែរក្សាគុណភាព។


In [13]:
!pip install -q foundry-local-sdk openai

### ការពន្យល់: ការនាំចូលមូលដ្ឋាន
នាំចូល regex សម្រាប់សម្គាល់ចេតនា, Foundry Local manager សម្រាប់ការភ្ជាប់តាមឈ្មោះជំនួស, និង OpenAI client សម្រាប់ការបញ្ចប់សន្ទនា.


In [14]:
import re
from foundry_local import FoundryLocalManager
from openai import OpenAI

### ការពិពណ៌នា៖ កាតាឡុកសមត្ថភាព និងច្បាប់
កំណត់កាតាឡុកដែលយល់ដឹងពីសមត្ថភាព និងច្បាប់ regex ដែលប្រើដើម្បីផ្គូផ្គងអត្ថបទនៃជំហានទៅកាន់ប្រភេទចេតនា (កូដ, សង្ខេប, ចាត់ថ្នាក់, ទូទៅ). តម្លៃអាទិភាពទាបបំផុតនឹងត្រូវបានជ្រើសនៅពេលមានម៉ូដែលច្រើនគាំទ្រចេតនា។


In [15]:
CATALOG = {
 'phi-4-mini': {'capabilities':['general','summarize'],'priority':2},
 'qwen2.5-coder-7b': {'capabilities':['code','refactor'],'priority':1},
 'qwen2.5-0.5b': {'capabilities':['classification','fast'],'priority':3},
}
RULES = [
 (re.compile('code|refactor|function', re.I), 'code'),
 (re.compile('summari|abstract|tl;dr', re.I), 'summarize'),
 (re.compile('classif|category|label', re.I), 'classification'),
]

### ការពន្យល់: បំណង, ការជ្រើសម៉ូឌែល និង ជំនួយការសន្ទនា
ផ្តល់:
- `detect_intent` សម្រាប់ចាត់ថ្នាក់ដោយផ្អែកលើ regex.
- `pick_model` ដើម្បីជ្រើសឈ្មោះជំនួសដែលល្អបំផុត ដោយផ្អែកលើសមត្ថភាព + អាទិភាព.
- `chat` ជា wrapper សម្រួល ដែលបង្កើត client ម្នាក់សម្រាប់រាល់ឈ្មោះជំនួស ហើយត្រឡប់មកនូវការឆ្លើយតបមួយជុំតែមួយ.


In [16]:
def detect_intent(text: str):
    """Return an intent label based on simple regex rules; falls back to 'general'."""
    for pat, intent in RULES:
        if pat.search(text):
            return intent
    return 'general'

def pick_model(intent: str) -> str:
    """Pick the best model for an intent using capability match first, then priority."""
    ranked = []
    for name, meta in CATALOG.items():
        ranked.append((name, intent in meta['capabilities'], meta['priority']))
    # Sort: capability match (True first), then lower priority value
    ranked.sort(key=lambda t: (not t[1], t[2]))
    return ranked[0][0]

def chat(alias: str, content: str, temp: float = 0.4) -> str:
    """Simple helper to send a single user message to a Foundry Local model via OpenAI client."""
    m = FoundryLocalManager(alias)
    client = OpenAI(base_url=m.endpoint, api_key=m.api_key or 'not-needed')
    mid = m.get_model_info(alias).id
    resp = client.chat.completions.create(
        model=mid,
        messages=[{'role': 'user', 'content': content}],
        max_tokens=180,
        temperature=temp,
    )
    return resp.choices[0].message.content

### ការពន្យល់: អនុគមន៍លំហូរដំណើរការ​ច្រើនជំហាន
អនុវត្តលំហូរដំណើរការ៖ រៀបចំផែនការ → វិភាគជំហាន → អនុវត្តរៀងរាល់ជំហានដោយការបញ្ជូនតាមផ្លូវផ្អែកលើចេតនា → លៃតម្រូវលទ្ធផលរួម។ ត្រឡប់ dict ដែលមានរចនាសម្ព័ន្ធសម្រាប់ពិនិត្យឬការប៉ាន់ប្រមាណ។


In [17]:
def pipeline(task: str):
    """Multi-step pipeline: plan → execute steps → refine final answer.

    Returns dict with keys: plan (raw plan text), steps (list of tuples), final (refined answer).
    Each step tuple: (index, step_text, step_result, model_alias_used)
    """
    # 1. Plan
    plan_alias = pick_model('general')
    plan_prompt = (
        "Break the task into 3 concise, actionable steps (no extra commentary).\n"
        f"Task: {task}"
    )
    plan = chat(plan_alias, plan_prompt)

    # 2. Parse steps (robust to numbering or bullet styles)
    raw_lines = [l.strip() for l in plan.splitlines() if l.strip()]
    steps = []
    for line in raw_lines:
        cleaned = re.sub(r'^\d+[).:-]?\s*', '', line)  # remove leading numbering
        cleaned = re.sub(r'^[-*]\s*', '', cleaned)      # remove bullet markers
        if cleaned:
            steps.append(cleaned)
        if len(steps) == 3:
            break
    if not steps:
        steps = [task]

    # 3. Execute steps
    outputs = []
    for idx, step in enumerate(steps, 1):
        intent = detect_intent(step)
        exec_alias = pick_model(intent)
        exec_prompt = (
            f"Execute step {idx} for the overall task.\n"
            f"Overall task: {task}\n"
            f"Step {idx}: {step}\n"
            "Return a concise result focusing only on the step objective."
        )
        result = chat(exec_alias, exec_prompt)
        outputs.append((idx, step, result, exec_alias))

    # 4. Refine final answer
    refine_alias = pick_model('summarize')
    combined = "\n\n".join(
        f"Step {idx} ({alias}) Output:\n{res}" for idx, _step, res, alias in outputs
    )
    refine_prompt = (
        "You are a senior assistant. Synthesize these step outputs into a cohesive final answer.\n"
        "Ensure clarity, avoid repetition, and highlight improvements or key insights if relevant.\n\n"
        f"Task: {task}\n\n"
        f"Step Outputs:\n{combined}"
    )
    final_ans = chat(refine_alias, refine_prompt)

    return {"plan": plan, "steps": outputs, "final": final_ans}

### ពន្យល់: រត់ភារកិច្ចឧទាហរណ៍
អនុវត្តបណ្តាញដំណើរការពេញលេញលើភារកិច្ចដែលផ្ដោតលើការកែសម្រួល ដើម្បីបង្ហាញពីគោលបំណងចម្រូងចម្រាស (កូដ + សង្ខេប)។ Result dict បង្ហាញផែនការដើម ផលលទ្ធផលអនុវត្តតាមជំហានៗ ជាមួយឈ្មោះជំនួសម៉ូឌែលដែលបានជ្រើស និងចម្លើយដែលបានសម្រួលចុងក្រោយ។


In [18]:
result = pipeline('Generate a refactored version of a slow Python loop and summarize performance gains.')
result

{'plan': '1. Profile the existing slow Python loop to identify bottlenecks.\n2. Refactor the loop using optimized techniques (e.g., list comprehensions, map, or itertools).\n3. Compare the performance of the refactored loop with the original using a benchmarking tool and summarize the gains.',
 'steps': [(1,
   'Profile the existing slow Python loop to identify bottlenecks.',
   "To execute step 1, you would use a profiling tool like `cProfile` in Python to analyze the performance of the existing slow loop. Here's an example of how you might do this:\n\n```python\nimport cProfile\n\ndef slow_loop():\n    # Example of a slow loop\n    result = []\n    for i in range(1000000):\n        result.append(i * i)\n    return result\n\n# Profile the slow loop\ncProfile.run('slow_loop()', 'profile_stats')\n\n# To see the results, you can use pstats module\nimport pstats\np = pstats.Stats('profile_stats')\np.sort_stats('cumulative').print_stats()\n```\n\nThis code will run the `slow_loop` function

### ពន្យល់: បង្ហាញវត្ថុលទ្ធផល

បង្ហាញលទ្ធផលដែលមានរចនាសម្ព័ន្ធនៃលំនាំដំណើរការ សម្រាប់ការត្រួតពិនិត្យយ៉ាងលឿន ឬសម្រាប់ការវាយតម្លៃបន្ទាប់ (ឧទាហរណ៍ ការវាស់គុណភាពនៃជំហាន ឬប្រសិទ្ធភាពនៃការកែលម្អ).


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាកម្មបកប្រែដោយ AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយ៉ាងណាក៏ដោយ ខណៈដែលយើងព្យាយាមឲ្យមានភាពត្រឹមត្រូវ សូមដឹងថាការបកប្រែដោយស្វ័យប្រវត្តិស្វែងរកអាចមានកំហុស ឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមនៅក្នុងភាសាដើមគួរត្រូវបានពិចារណាថាជាប្រភពដែលទុកចិត្តបាន។ សម្រាប់ព័ត៌មានសំខាន់ អាចមានការផ្តល់យោបល់ឱ្យបកប្រែដោយមនុស្សជំនាញ។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសដែលកើតឡើងពីការប្រើប្រាស់ការបកប្រែនេះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
